# 🚀 02. Train Vision-Only Student Model (Phase 2 - CMKD)

> **Project:** Durian Ripeness Classification via Cross-Modal Knowledge Distillation (CMKD)  
> **Lead:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Reference:** `specification/project_specification.pdf` (Section 1.3.3) & `tmp_student.md`  
> **Notebook Role:**
1. Mount Google Drive & verify GPU availability.
2. `git clone` / `git pull` the latest source code from GitHub.
3. Link dataset (`data/final`), Teacher checkpoint (`checkpoints/teachermodal/best_teacher.pth`), and create target folders (`checkpoints/studentmodal`, `results_student`) on Google Drive.
4. Launch 2-phase distillation training (`train_student.py`) and export the benchmark comparison table.

In [7]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & CHECK GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
else:
    print("ℹ️ Running in Local environment.")

!nvidia-smi

✅ Google Drive mounted successfully!
Fri Nov 20 19:41:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------

In [8]:
# ==============================================================================
# 2. SYNC LATEST CODE FROM GITHUB
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-classification"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        if BRANCH:
            !git clone -b $BRANCH $REPO_URL $REPO_DIR
        else:
            !git clone $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Remove temporary repo symlinks (without touching Drive files) to prevent git pull conflicts
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        if BRANCH:
            !git checkout -f $BRANCH
            !git reset --hard origin/$BRANCH
        else:
            !git reset --hard origin/$(git rev-parse --abbrev-ref HEAD)
else:
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Working in project root directory: {os.getcwd()}")

/content
/content/durian-ripeness-classification
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 45 (delta 18), reused 45 (delta 18), pack-reused 0 (from 0)
Unpacking objects: 100% (45/45), 1.27 MiB | 4.67 MiB/s, done.
From https://github.com/Cocobaut/durian-ripeness-classification
   ddb1ea7..60b3fe0  cocobaut   -> origin/cocobaut
branch 'cocobaut' set up to track 'origin/cocobaut'.
Switched to a new branch 'cocobaut'
HEAD is now at 60b3fe0 [UPDATE] train teacher
✅ Working in project root directory: /content/durian-ripeness-classification


In [9]:
# ==============================================================================
# 3. FAST DEPENDENCY INSTALLATION WITH `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Installed all dependencies successfully!")

Using Python 3.13.16 environment at: /usr
Checked 22 packages in 104ms
✅ Installed all dependencies successfully!


In [10]:
# ==============================================================================
# 4. CONFIGURE CHECKPOINT & DATASET LINKS FROM GOOGLE DRIVE
# ==============================================================================
from pathlib import Path

if IN_COLAB:
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_RESULTS_TEACHER = DRIVE_PROJECT_ROOT / "results_teacher"
    DRIVE_RESULTS_STUDENT = DRIVE_PROJECT_ROOT / "results_student"

    # Pre-create studentmodal & results_student folders on Google Drive for real-time persistence
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)
    (DRIVE_CHECKPOINTS / "studentmodal").mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS_STUDENT.mkdir(parents=True, exist_ok=True)

    # Symlink repo directories to Google Drive so outputs persist across Colab disconnects
    !rm -rf checkpoints/unimodal checkpoints/teachermodal checkpoints/studentmodal
    !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'studentmodal')}" checkpoints/studentmodal
    !ln -sfn "{str(DRIVE_RESULTS_STUDENT)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    if DRIVE_RESULTS_TEACHER.exists():
        !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results
        !ln -sfn "{str(DRIVE_RESULTS_TEACHER)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/studentmodal").mkdir(parents=True, exist_ok=True)
print("✅ Verifying configured paths:")
print(" - Teacher ckpt   :", Path("checkpoints/teachermodal/best_teacher.pth").exists(), "->", Path("checkpoints/teachermodal").resolve())
print(" - Studentmodal   :", Path("checkpoints/studentmodal").resolve())
print(" - Results student:", Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())

✅ Verifying configured paths:
 - Teacher ckpt   : True -> /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Studentmodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/studentmodal
 - Results student: /content/drive/MyDrive/durian-ripeness-cmkd/results_student
 - Manifest CSV   : True


In [11]:
# ==============================================================================
# 5. RUN MULTI-VIEW VISION-ONLY STUDENT DISTILLATION TRAINING (PHASE 2 - CMKD)
# ==============================================================================
# Improved CMKD recipe (v2):
#  - L_feat aligns with the Teacher's VISION-branch embedding (learnable from RGB) instead of f_mm,
#    using cosine + relational (in-batch similarity structure) distance, with a small weight (gamma 0.3).
#  - L_KD (T=4) only on fruits the Teacher classifies correctly (do not copy Teacher mistakes).
#  - Gentler Phase 2 (backbone LR 5e-6), EMA weights, checkpoint selected by val loss (24 val fruits -> F1 is noisy).
#  - Validation + final evaluation use the same 3-view TTA as the unimodal Vision baseline (fair comparison).
#  - --fresh: ignore any old last_student.pth (only the Student is retrained; Teacher/unimodal ckpts are reused).
!python workspaces/member1_vision_lead/04_train_teacher_and_student/train_student.py \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --teacher-ckpt checkpoints/teachermodal/best_teacher.pth \
    --student-backbone convnextv2_base \
    --save-path checkpoints/studentmodal/best_student.pth \
    --results-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student \
    --alpha 1.0 \
    --beta 1.0 \
    --gamma 0.3 \
    --temperature 4.0 \
    --feat-target vision \
    --feat-loss-type cosine_relational \
    --kd-weighting teacher_correct \
    --loss-type cost_sensitive_focal \
    --focal-gamma 2.0 \
    --cost-weight 0.5 \
    --batch-size 4 \
    --accum-steps 2 \
    --epochs 30 \
    --warmup-epochs 5 \
    --lr 5e-4 \
    --lr-backbone 5e-6 \
    --min-lr 1e-6 \
    --patience 8 \
    --ema-decay 0.99 \
    --select-metric val_loss \
    --eval-test \
    --fresh \
    --device cuda

[2026-11-20 19:41:32] ==============================================================================
[2026-11-20 19:41:32] TRAINING VISION-ONLY STUDENT VIA CMKD (STAGE 2)
[2026-11-20 19:41:32] Device: cuda | AMP: True (torch.bfloat16) | batch 4 x accum 2
[2026-11-20 19:41:32] Student backbone : convnextv2_base.fcmae_ft_in1k
[2026-11-20 19:41:32] Distill weights  : alpha=1.0 (CE) | beta=2.0 (KD, T=3.0) | gamma=1.0 (Feat: cosine)
[2026-11-20 19:41:32] Best student ckpt: /content/durian-ripeness-classification/checkpoints/studentmodal/best_student.pth
[2026-11-20 19:41:32] ==============================================================================
[2026-11-20 19:41:32] Loading Frozen Teacher from: /content/durian-ripeness-classification/checkpoints/teachermodal/best_teacher.pth
/content/durian-ripeness-classification/src/datasets/sensor_dataset.py:132: UserWarning: Failed to load /content/durian-ripeness-classification/workspaces/member3_sensor/02_train_unimodal_sensor/results/sensor_p

In [12]:
# ==============================================================================
# 6. DISPLAY BENCHMARK COMPARISON TABLE (evaluation already run by --eval-test in cell 5)
# ==============================================================================
# To re-evaluate a checkpoint separately (same 3-view TTA as the unimodal baseline):
# !python workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py \
#     --checkpoint checkpoints/studentmodal/best_student.pth --split test \
#     --output-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student

comp_md = Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student/benchmark_comparison.md")
if comp_md.is_file():
    print("\n" + comp_md.read_text(encoding="utf-8"))
print("✅ All Student evaluation results saved at:", comp_md.parent.resolve())

[2026-11-20 20:09:04] [INFO] [eval_teacher]: Initialized TeacherEvaluator on device: cuda (input_mode=vision)
[2026-11-20 20:09:04] [INFO] [eval_teacher]: Evaluation results will be saved to: /content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
[2026-11-20 20:09:04] [INFO] [eval_teacher]: === STARTING FULL TEACHER EVALUATION PIPELINE ===
[2026-11-20 20:09:04] [INFO] [eval_teacher]: model=vision_student | split=test | input_mode=vision | strategy=mean_prob
[2026-11-20 20:09:04] [INFO] [eval_teacher]: Starting sample-level evaluation (mask_modality=None)...
[2026-11-20 20:09:11] [INFO] [eval_teacher]: Evaluated 24 samples, accuracy = 0.5000
[2026-11-20 20:09:11] [INFO] [eval_teacher]: Aggregating fruit-level predictions (fruit_id) using strategy: mean_prob...
[2026-11-20 20:09:11] [INFO] [eval_teacher]: Aggregated 24 fruits, fruit-level accuracy = 0.5000
[2026-11-20 20:09:12] [INFO] [eval_teacher]: Starting Modality Ablation 